# 01 — Pitcher arsenal

Pull one pitcher's season of Statcast data, check it, and build his arsenal table two ways: pandas and DuckDB SQL.

**Change the pitcher or season in the next cell**, then run all cells (Run All at the top of VS Code).

If Baseball Savant is down or slow, set `USE_SAMPLE = True` to run everything on fictional data instead.

In [ ]:
PITCHER = ("skubal", "tarik")   # (last, first)
SEASON = 2025
USE_SAMPLE = False               # True = fictional data, no internet needed
MIN_N = 20                       # pitch types thrown fewer times than this are left out

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from baseball.arsenal import ARSENAL_SQL, arsenal, arsenal_sql
from baseball.sample import fictional_pitcher
from baseball.statcast import lookup_mlbam, pitcher_season

pd.set_option("display.max_columns", 50)

## 1. Load the data
The first real pull takes a minute or two. After that it loads from `data/raw/` instantly.

In [ ]:
if USE_SAMPLE:
    pitches = fictional_pitcher()
    label = "Fictional pitcher (sample data)"
else:
    mlbam = lookup_mlbam(*PITCHER)
    pitches = pitcher_season(mlbam, SEASON)
    label = f"{PITCHER[1].title()} {PITCHER[0].title()}, {SEASON}"

print(label)
print(f"{len(pitches):,} pitches")
pitches.head()

## 2. Check the data before trusting it
Row count, pitch types, and missing values in the columns the table uses. A pitch type with a handful of pitches is often a mislabel.

In [ ]:
needed = ["pitch_type", "release_speed", "release_spin_rate", "pfx_x", "pfx_z", "description", "p_throws"]
print("Missing values per column:")
print(pitches[needed].isna().sum().to_string())
print()
print("Pitches by type:")
print(pitches["pitch_type"].value_counts(dropna=False).to_string())

## 3. Arsenal table — pandas

In [ ]:
table = arsenal(pitches, min_n=MIN_N)
table

## 4. The same table — DuckDB SQL
The query below is what `arsenal_sql` runs. The numbers should match the pandas table (rounding can differ by 0.1, since pandas rounds a trailing 5 to the even digit).

In [ ]:
print(ARSENAL_SQL)
arsenal_sql(pitches, min_n=MIN_N)

## 5. Movement plot
Each dot is one pitch, from the pitcher's point of view: right = arm side, up = more rise than gravity alone.

In [ ]:
m = pitches.dropna(subset=["pitch_type", "pfx_x", "pfx_z"]).copy()
m = m[m["pitch_type"].isin(table["pitch_type"])]
m["ivb_in"] = m["pfx_z"] * 12
m["hb_arm_in"] = np.where(m["p_throws"].eq("R"), -m["pfx_x"], m["pfx_x"]) * 12

fig, ax = plt.subplots(figsize=(7, 7))
for (code_, name), g in m.groupby(["pitch_type", "pitch_name"]):
    ax.scatter(g["hb_arm_in"], g["ivb_in"], s=8, alpha=0.35, label=f"{name} ({len(g)})")
ax.axhline(0, color="grey", lw=0.8)
ax.axvline(0, color="grey", lw=0.8)
ax.set_xlim(-25, 25)
ax.set_ylim(-25, 25)
ax.set_aspect("equal")
ax.set_xlabel("Horizontal break, arm side + (in)")
ax.set_ylabel("Induced vertical break (in)")
ax.set_title(label)
ax.legend(loc="lower left", fontsize=8, markerscale=2)
plt.show()

## Next ideas
- Run it for a second pitcher and compare the two tables side by side.
- Split the table by batter handedness (`stand` column) to see how he attacks lefties vs righties.
- Fastball velocity by game (`game_date`) to spot a velo dip or gain across the season.